# 🧒 AI Storytellers for Kids

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/editor/creative/ai_storyteller_for_kids.ipynb)

### An AI-Powered Educational Video Generator for Kids

Ever wished you could create fun, educational videos for kids with just a single topic?

**ELI5** takes a topic like *"explain the solar system"* and magically transforms it into a complete animated video with:
- 🎙️ Kid-friendly voiceover narration
- 🎵 Fun background music  
- 🌌 Loopable animated backgrounds
- 🎬 Timed subject videos (cartoons of planets, stars, etc.)
- 📝 Animated captions that kids can follow along

All powered by **VideoDB's Editor SDK** — no external tools, pure automation magic! ✨

---

## 🚀 Setup

In [3]:
!pip install -q videodb

In [4]:
import os
import math
from getpass import getpass

import videodb
from videodb import play_stream
from videodb.editor import (
    Timeline, Track, Clip,
    VideoAsset, AudioAsset, ImageAsset, TextAsset, CaptionAsset,
    Fit, Position, Offset, Transition, Filter,
    Font, Border, Shadow, Background, Alignment,
    CaptionAnimation, CaptionAlignment, CaptionBorderStyle,
    FontStyling, Positioning, BorderAndShadow
)

print("✅ All imports ready!")

✅ All imports ready!


In [5]:
# Connect to VideoDB
api_key = getpass("🔑 Enter your VideoDB API Key: ")
os.environ["VIDEO_DB_API_KEY"] = api_key

conn = videodb.connect()
coll = conn.get_collection()

print("✅ Connected to VideoDB!")

🔑 Enter your VideoDB API Key: ··········
✅ Connected to VideoDB!


---

## 🎯 Step 1: Pick a Topic

What do you want to learn today? Type anything — the solar system, dinosaurs, why the sky is blue... we'll explain it like you're five! 🧒

In [6]:
# 🎯 Your topic goes here!
topic = "Explain the solar system"

print(f"🎬 Let's make a fun video about: '{topic}'")

🎬 Let's make a fun video about: 'Explain the solar system'


---

## ✍️ Step 2: Generate the Script & Title

Time to ask our AI friend to write a super simple, fun script that even a 5-year-old can understand! We'll also get a catchy title for our video. 🌟

In [7]:
import json
# Generate kid-friendly script and catchy title
print("📝 Generating script...")

script_prompt = f"""You are a friendly children's educational content writer.

Write a fun, simple explanation about: "{topic}"

Requirements:
- Target audience: 5-year-old children
- Length: 250 words
- Use simple words and short sentences
- Make it fun and engaging with analogies kids understand
- No complex terminology
- Sound enthusiastic and friendly!

Return a JSON with:
- "title": A fun, catchy title for the video (max 5 words)
- "script": The complete narration script (just the text to be spoken, no stage directions)

Example format:
{{"title": "The Amazing Sun!", "script": "Hey friends! Today we're going to learn about..."}}
"""

script_response = coll.generate_text(
    prompt=script_prompt,
    model_name="pro",
    response_type="json"
)

# Ensure script_response is a dictionary before proceeding
if isinstance(script_response, str):
    script_response = json.loads(script_response)

# Adjusting for nested 'output' key if present
if 'output' in script_response and isinstance(script_response['output'], dict):
    video_title = script_response["output"]["title"]
    video_script = script_response["output"]["script"]
else:
    video_title = script_response["title"]
    video_script = script_response["script"]

print(f"✅ Script generated!")
print(f"\n🎬 Title: {video_title}")
print(f"\n📜 Script:\n{video_script}")

📝 Generating script...
✅ Script generated!

🎬 Title: Space Playground

📜 Script:
Hey friends! Today we will take a fun trip. We will learn about the solar system. The solar system is like a big family. The Sun is the biggest member. It is like a warm lamp. It gives light and heat. The planets are like toys that go around the lamp. They spin and travel in circles. The closest planet to the Sun is small and hot. A little farther is a planet that looks blue and has wind. Next comes our home. Earth is our planet. It has water, land, and air. We live on Earth with plants and animals. Mars is red like a dusty ball. Farther out are big gas giants. They are very big like giant balloons. One has rings like a hula hoop. The planets are different sizes and colors. They all dance around the Sun. There are also tiny pieces called rocks and ice. Some make bright streaks when they zoom by. The Sun keeps all the planets together with a gentle pull. Imagine holding a ball and spinning a string. That is

---

## 🎨 Step 3: Design Background Vibes

Now let's ask AI to come up with prompts for our background video and music. We want something dreamy and loopable — no distracting subjects, just vibes! ✨

In [8]:
# 🎨 Step 3: Design Background Vibes (BALANCED - Fun but Relevant!)
print("🎨 Designing FUN background vibes...")

bg_prompt = f"""Based on this children's educational video topic: "{topic}"

Create prompts for background media that will play behind narration.

Requirements for background VIDEO prompt:
- COLORFUL, BRIGHT, and VIBRANT (engaging for kids!)
- MUST be thematically related to "{topic}"
- Use abstract/simplified versions of topic elements (NOT realistic, cartoonish style)
- Think: animated patterns, flowing shapes, gradient backgrounds with theme colors
- Examples:
  * For space topics: colorful stars, abstract planets, twinkling cosmic patterns
  * For ocean topics: flowing waves, bubbles, sea-themed gradients
  * For nature topics: leaves, flowers, trees in motion - all cartoonish
  * For science topics: molecules, atoms, light beams - simplified and colorful
- Bright primary and secondary colors: blues, greens, yellows, oranges, purples
- Smooth, loopable motion (5 seconds)
- Kid-friendly but NOT chaotic or random
- Educational vibe meets fun aesthetics

Requirements for background MUSIC prompt:
- UPBEAT, CATCHY, and FUN!
- Fast-paced rhythm that kids can bounce to
- Playful instruments: ukulele, clapping, percussion, bright synths
- Major key, happy and energetic
- Think nursery rhyme energy meets pop music catchiness
- Instrumental only, no lyrics
- The kind of music that makes you tap your feet!

Return JSON with:
- "background_video_prompt": A detailed prompt for generating the vibrant, topic-relevant background video
- "music_prompt": A prompt for generating upbeat background music

Example for "explain the solar system":
{{"background_video_prompt": "Colorful abstract space background with flowing starry patterns, simplified cartoon planets orbiting in soft motion, bright gradient from deep purple to electric blue, twinkling stars and cosmic dust particles, playful and educational vibe, seamless loop, vibrant colors", "music_prompt": "Upbeat playful ukulele melody with hand claps and percussion, bright cheerful children's music, fast tempo, catchy rhythm, major key, bouncy and fun, toe-tapping energy"}}
"""

bg_response = coll.generate_text(
    prompt=bg_prompt,
    model_name="pro",
    response_type="json"
)

# Handle nested output structure
if isinstance(bg_response, dict) and "output" in bg_response:
    background_video_prompt = bg_response["output"]["background_video_prompt"]
    music_prompt = bg_response["output"]["music_prompt"]
else:
    background_video_prompt = bg_response["background_video_prompt"]
    music_prompt = bg_response["music_prompt"]

print(f"✅ FUN & RELEVANT background prompts ready!")
print(f"\n🎬 Video Prompt: {background_video_prompt}")
print(f"\n🎵 Music Prompt: {music_prompt}")

🎨 Designing FUN background vibes...
✅ FUN & RELEVANT background prompts ready!

🎬 Video Prompt: Create a colorful, bright, and vibrant 2D cartoonish background themed around the solar system. Use simplified, abstract planet shapes (circles with soft rings), stylized suns, moons, and comet streaks — not realistic, very playful and kid-friendly. Palette: bold primary and secondary colors (bright blues, greens, sunny yellows, warm oranges, lively purples, pink accents). Background is a smooth gradient field (deep purple to electric blue to teal) with flowing shapes and layered rounded waves. Add repeating, simplified orbit lines, soft parallax of 3–4 planet shapes gently orbiting on slow, even curves. Include small twinkling star icons, colorful cosmic dust particles, and occasional pulsing halos around planets — all simplified and geometric. Animation style: flat vector, soft drop shadows, rounded edges, clean outlines, and consistent stroke width. Motion: very smooth, loopable 5-second 

---

## 🎬 Step 4: Generate All the Assets!

This is where the magic happens! We're generating:
1. 🎙️ **Voiceover** from our script
2. 🎵 **Background music** (10 seconds, loopable)
3. 🌌 **Background video** (5 seconds, loopable)

Let the AI do its thing... ⏳

In [9]:
# Generate voiceover from script
print("🎙️ Generating voiceover...")
voiceover = coll.generate_voice(
    text=video_script,
    voice_name="Default"
)
print(f"✅ Voiceover ready! ID: {voiceover.id}, Length: {voiceover.length}s")

🎙️ Generating voiceover...
✅ Voiceover ready! ID: a-z-019f9041-b56f-7472-aff0-f7a402f2146e, Length: 85.263673s


In [11]:
# Generate background music (10 seconds, loopable)
print("🎵 Generating background music...")
bg_music = coll.generate_music(
    prompt=music_prompt,
    duration=10
)
print(f"✅ Background music ready! ID: {bg_music.id}, Length: {bg_music.length}s")

🎵 Generating background music...
✅ Background music ready! ID: a-z-019f9043-7c3e-7573-836e-00e786c94995, Length: 32.768229s


In [12]:
# Generate background video (5 seconds, loopable)
print("🌌 Generating background video...")
bg_video = coll.generate_video(
    prompt=background_video_prompt,
    duration=5
)
print(f"✅ Background video ready! ID: {bg_video.id}")

🌌 Generating background video...
✅ Background video ready! ID: m-z-019f9045-1a1a-7f92-95be-3b235ace0a2b


---

## 📝 Step 5: Get Timestamped Transcript

Timed transcript segments help our AI figure out exactly *when* to show subject videos. They also keep our custom captions in sync! 🎯

In [13]:
# Generate and fetch timestamped transcript
print("📝 Generating transcript with timestamps...")

voiceover.generate_transcript()
transcript = voiceover.get_transcript(segmenter=videodb.Segmenter.time, length=5)

# Calculate total video duration (intro + voiceover + outro)
voiceover_length_float = float(voiceover.length)
intro_duration = 5  # Title screen
outro_duration = 5  # Happy Learning screen
total_duration = intro_duration + voiceover_length_float + outro_duration

print(f"✅ Transcript ready! {len(transcript)} segments")
print(f"\n⏱️ Video breakdown:")
print(f"   - Intro: {intro_duration}s")
print(f"   - Voiceover: {voiceover_length_float}s")
print(f"   - Outro: {outro_duration}s")
print(f"   - Total: {total_duration}s")

import json
print(json.dumps(transcript, indent=2))

📝 Generating transcript with timestamps...
✅ Transcript ready! 18 segments

⏱️ Video breakdown:
   - Intro: 5s
   - Voiceover: 85.263673s
   - Outro: 5s
   - Total: 95.263673s
[
  {
    "end": 5.0,
    "start": 0,
    "text": "Hey friends, today we will take a fun trip, we will learn about the solar system."
  },
  {
    "end": 10.0,
    "start": 0.0,
    "text": " The solar system is like a big family."
  },
  {
    "end": 15.0,
    "start": 5.0,
    "text": " The sun is the biggest member, it is like a warm warm lamp, it gives light and heat, the planets are like toys that go around the"
  },
  {
    "end": 20.0,
    "start": 15.0,
    "text": " lamp, they spin and travel in circles."
  },
  {
    "end": 25.0,
    "start": 15.0,
    "text": " The closest planet to the sun is small on hot. A little farther is a planet that looks blue and has wind."
  },
  {
    "end": 30.0,
    "start": 20.0,
    "text": " Next Next comes our home. Earth is our planet. It has water, land and air."
  }

---

## 🎥 Step 6: Preview — Base Video with Title & Outro

Let's see what we've got so far! This preview includes:
- 🌌 Looped background video
- 🎵 Looped background music (soft volume)
- 🎙️ Voiceover narration
- 📝 Auto-generated animated captions
- 🎬 Intro title card (5 seconds)
- 👋 Outro "Happy Learning!" card (5 seconds)

No subject videos yet — that's the next step!

In [23]:
# 📝 Configure reusable auto-generated captions
caption_asset = CaptionAsset(
    src="auto",
    animation=CaptionAnimation.karaoke,
    primary_color="&H00FFFFFF",    # White
    secondary_color="&H0000FFFF",  # Yellow
    back_color="&H80000000",
    position=Positioning(
        alignment=CaptionAlignment.bottom_center,
        margin_l=20,
        margin_r=20,
        margin_v=30
    ),
    font=FontStyling(
        name="Clear Sans",
        size=52,
        bold=True
    ),
    border=BorderAndShadow(
        style=CaptionBorderStyle.outline_and_shadow,
        outline=4,
        outline_color="&H00000000",  # Black
        shadow=2
    )
)

print("✅ Reusable caption style ready!")

✅ Reusable caption style ready!


/tmp/ipykernel_2363/118926981.py:2: UserWarning: CaptionAsset(src='auto'): the video must be indexed (e.g. video.index_spoken_words()) for captions to be generated.
  caption_asset = CaptionAsset(


In [24]:
import math

# 🎬 COMPLETE VIDEO with Intro, Outro, and Auto-Generated Captions
print("🎬 Building COMPLETE video...")

# Calculate total duration and loops
total_duration = intro_duration + voiceover_length_float + outro_duration

# Use the actual length of the background video asset
bg_video_actual_length = bg_video.length

video_loops = math.ceil(total_duration / bg_video_actual_length)
music_loops = math.ceil(total_duration / 10)

print(f"⏱️ Total: {total_duration}s (intro:{intro_duration} + vo:{voiceover_length_float} + outro:{outro_duration})")

timeline = Timeline(conn)
timeline.background = "#0a0a1a"
timeline.resolution = "1280x720"

# Track 1: Looped background video
bg_video_track = Track()
for i in range(video_loops):
    start = i * bg_video_actual_length  # Use actual length
    remaining = total_duration - start
    dur = min(bg_video_actual_length, remaining)  # Use actual length

    if dur <= 0:  # Ensure positive duration
        break

    bg_video_track.add_clip(start=start, clip=Clip(asset=VideoAsset(id=bg_video.id, volume=0), duration=dur, fit=Fit.cover, opacity=0.7))

# Track 2: Looped background music
bg_music_track = Track()
for i in range(music_loops):
    start = i * 10
    remaining = total_duration - start
    dur = min(10, remaining)
    bg_music_track.add_clip(start=start, clip=Clip(asset=AudioAsset(id=bg_music.id, volume=0.5), duration=dur))

# Track 3: Voiceover and captions (start after intro)
vo_track = Track()
vo_track.add_clip(start=intro_duration, clip=Clip(asset=AudioAsset(id=voiceover.id, volume=1.0), duration=voiceover_length_float))
vo_track.add_clip(start=intro_duration, clip=Clip(asset=caption_asset, duration=voiceover_length_float))

# Track 4: Title card (intro) - BIGGER, QUICKSAND FONT
title_track = Track()
title_track.add_clip(
    start=0,
    clip=Clip(
        asset=TextAsset(
            text=video_title,
            font=Font(
                family="Quicksand",
                size=96,              # Bigger!
                color="#FFD700"       # Gold
            ),
            border=Border(
                color="#000000",
                width=6               # Thick black border
            )
            # No shadow!
        ),
        duration=intro_duration,
        position=Position.center,
        transition=Transition(in_="fade", out="fade", duration=0.8)
    )
)

# Track 5: Outro card - BIGGER, QUICKSAND FONT
outro_track = Track()
outro_track.add_clip(
    start=intro_duration + voiceover_length_float,
    clip=Clip(
        asset=TextAsset(
            text="Happy Learning!",
            font=Font(
                family="Quicksand",
                size=88,              # Bigger!
                color="#90EE90"       # Light green
            ),
            border=Border(
                color="#000000",
                width=6               # Thick black border
            )
            # No shadow!
        ),
        duration=outro_duration,
        position=Position.center,
        transition=Transition(in_="fade", out="fade", duration=0.8)
    )
)

# Add all tracks
timeline.add_track(bg_video_track)
timeline.add_track(bg_music_track)
timeline.add_track(vo_track)
timeline.add_track(title_track)
timeline.add_track(outro_track)

print("✅ Timeline built!")

# Generate stream
print("\n🚀 Generating stream...")
stream = timeline.generate_stream()
play_stream(stream)

🎬 Building COMPLETE video...
⏱️ Total: 95.263673s (intro:5 + vo:85.263673 + outro:5)
✅ Timeline built!

🚀 Generating stream...


---

## 🎯 Step 7: Generate Subject Video Prompts

Time to bring the fun visuals! Our AI will analyze the transcript and create video prompts for each pair of narration lines — giving us full visual coverage throughout the video! 🌍☀️🌙

We'll generate one cartoonish video clip for every 2 transcript segments, ensuring continuous visuals from start to finish.

In [17]:
# 🎯 Step 7: Generate Subject Video Prompts (1 per 2 Transcript Lines)
print("🎯 Generating video prompts for paired transcript segments...")

# Group transcript into pairs
transcript_pairs = []
for i in range(0, len(transcript), 2):
    pair = {
        "start": float(transcript[i]['start']),
        "end": float(transcript[i + 1]['end']) if i + 1 < len(transcript) else float(transcript[i]['end']),
        "text": transcript[i]['text'] + " " + (transcript[i + 1]['text'] if i + 1 < len(transcript) else "")
    }
    transcript_pairs.append(pair)

print(f"📊 Grouped {len(transcript)} lines into {len(transcript_pairs)} pairs")

subject_prompt = f"""You are creating visual prompts for a children's educational video about: "{topic}"

For each PAIR of narration lines below, create ONE video generation prompt that captures the essence of both lines.

Transcript pairs:
{json.dumps(transcript_pairs, indent=2)}

Requirements for each video prompt:
- Style: Cartoonish, whimsical, colorful, kid-friendly
- NO text in the videos
- NO complex actions
- Focus on a SINGLE subject that represents both lines
- Think fun and simple — like a storybook illustration come to life
- Slight motion is okay (spinning, floating, twinkling, bouncing)

Return a JSON object with a "videos" array containing ONE item per PAIR, in the SAME ORDER:
{{
    "videos": [
        {{"subject": "<what it shows>", "prompt": "<video generation prompt for pair 1>"}},
        {{"subject": "<what it shows>", "prompt": "<video generation prompt for pair 2>"}},
        ...
    ]
}}

You MUST return exactly {len(transcript_pairs)} items, one for each pair.
"""

subject_response = coll.generate_text(
    prompt=subject_prompt,
    model_name="pro",
    response_type="json"
)

video_prompts = subject_response['output']['videos']
if not isinstance(video_prompts, list) or len(video_prompts) != len(transcript_pairs):
    raise ValueError("Expected exactly one video prompt per transcript pair.")

# Combine prompts with paired transcript timestamps
subject_moments = []
for i, pair in enumerate(transcript_pairs):
    prompt_data = video_prompts[i]
    subject_moments.append({
        "start": pair['start'],
        "end": pair['end'],
        "text": pair['text'],
        "subject": prompt_data.get('subject', f'Scene {i+1}'),
        "prompt": prompt_data['prompt']
    })

print(f"✅ Generated {len(subject_moments)} video prompts for {len(transcript)} transcript lines!")
print("\n📋 Preview:")
for i, moment in enumerate(subject_moments[:3], 1):
    print(f"   {i}. [{moment['start']:.1f}s - {moment['end']:.1f}s] {moment['subject']}")
    print(f"      Text: {moment['text'][:60]}...")
    print(f"      Prompt: {moment['prompt'][:60]}...")
if len(subject_moments) > 3:
    print(f"   ... and {len(subject_moments) - 3} more")

🎯 Generating video prompts for paired transcript segments...
📊 Grouped 18 lines into 9 pairs
✅ Generated 9 video prompts for 18 transcript lines!

📋 Preview:
   1. [0.0s - 10.0s] Smiling sun with orbiting planet family
      Text: Hey friends, today we will take a fun trip, we will learn ab...
      Prompt: Cartoonish, whimsical, colorful scene of a big smiling sun a...
   2. [5.0s - 20.0s] Warm lamp-like sun with toy planets circling
      Text:  The sun is the biggest member, it is like a warm warm lamp,...
      Prompt: Playful illustration of a warm glowing sun that looks like a...
   3. [15.0s - 30.0s] Three simple planets: tiny hot, blue windy, and Earth
      Text:  The closest planet to the sun is small on hot. A little far...
      Prompt: Kid-friendly, colorful depiction of three planets in a row: ...
   ... and 6 more


---

## 🎨 Step 8: Generate Subject Videos

Now we generate one subject video per transcript pair! Each one is a 5-second cartoonish clip that will pop up during the narration. This might take a minute... ☕

In [18]:
# 🎨 Step 8: Generate Subject Videos (1 per 2 Transcript Lines)
print(f"🎨 Generating {len(subject_moments)} subject videos...")
print("=" * 50)

subject_videos = []

for i, moment in enumerate(subject_moments, 1):
    print(f"\n🎬 [{i}/{len(subject_moments)}] Generating: {moment['subject']}")
    print(f"   ⏱️ Coverage: {moment['start']:.1f}s - {moment['end']:.1f}s")

    subject_video = coll.generate_video(
        prompt=moment['prompt'],
        duration=5  # 5s video, will loop as needed
    )

    subject_videos.append({
        "video": subject_video,
        "start": moment['start'],
        "end": moment['end'],
        "subject": moment['subject']
    })

    print(f"   ✅ Done! ID: {subject_video.id}")

print("\n" + "=" * 50)
print(f"🎉 All {len(subject_videos)} subject videos generated!")
print("\n📋 Summary:")
for sv in subject_videos:
    duration = sv['end'] - sv['start']
    print(f"   - [{sv['start']:.1f}s - {sv['end']:.1f}s] ({duration:.1f}s) {sv['subject']} → {sv['video'].id}")

🎨 Generating 9 subject videos...

🎬 [1/9] Generating: Smiling sun with orbiting planet family
   ⏱️ Coverage: 0.0s - 10.0s
   ✅ Done! ID: m-z-019f905b-963a-7750-94fb-24c6139a96be

🎬 [2/9] Generating: Warm lamp-like sun with toy planets circling
   ⏱️ Coverage: 5.0s - 20.0s
   ✅ Done! ID: m-z-019f905c-f1f5-77c1-8e6f-bfb4b38a978d

🎬 [3/9] Generating: Three simple planets: tiny hot, blue windy, and Earth
   ⏱️ Coverage: 15.0s - 30.0s
   ✅ Done! ID: m-z-019f905e-5f8a-73a3-8a86-5a4ac6aacdfe

🎬 [4/9] Generating: Earth with nearby red Mars and a giant gas balloon planet
   ⏱️ Coverage: 25.0s - 40.0s
   ✅ Done! ID: m-z-019f905f-906f-7bb0-ae69-fb3462cd5d1c

🎬 [5/9] Generating: Ringed planet leading a dance of colorful planets and tiny rocks
   ⏱️ Coverage: 35.0s - 50.0s
   ✅ Done! ID: m-z-019f9060-c790-7893-97bb-3a7d88e83268

🎬 [6/9] Generating: Sun as a gentle anchor with a comet streaking by
   ⏱️ Coverage: 45.0s - 60.0s
   ✅ Done! ID: m-z-019f9062-0991-7500-a7f3-5522c03d1019

🎬 [7/9] Generat

---

## 🎬 Step 9: Final Composition!

This is it — the grand finale! We're adding all the subject videos as picture-in-picture overlays. They'll appear center-top so they don't cover the captions at the bottom.

Let's bring it all together! 🚀✨

In [25]:
# 🎬 FINAL VIDEO with Subject Videos (1 per 2 lines, looped) and Auto-Generated Captions
print("🎬 Building FINAL video with subject videos...")

# Calculate total duration and loops
total_duration = intro_duration + voiceover_length_float + outro_duration

# Use the actual length of the background video asset
bg_video_actual_length = bg_video.length

video_loops = math.ceil(total_duration / bg_video_actual_length)
music_loops = math.ceil(total_duration / 10)

print(f"⏱️ Total: {total_duration}s (intro:{intro_duration} + vo:{voiceover_length_float} + outro:{outro_duration})")

final_timeline = Timeline(conn)
final_timeline.background = "#0a0a1a"
final_timeline.resolution = "1280x720"

# Track 1: Looped background video
bg_video_track = Track()
for i in range(video_loops):
    start = i * bg_video_actual_length # Use actual length
    remaining = total_duration - start
    dur = min(bg_video_actual_length, remaining) # Use actual length

    if dur <= 0: # Ensure positive duration
        break

    bg_video_track.add_clip(start=start, clip=Clip(asset=VideoAsset(id=bg_video.id, volume=0), duration=dur, fit=Fit.cover, opacity=0.7))

# Track 2: Looped background music
bg_music_track = Track()
for i in range(music_loops):
    start = i * 10
    remaining = total_duration - start
    dur = min(10, remaining)
    bg_music_track.add_clip(start=start, clip=Clip(asset=AudioAsset(id=bg_music.id, volume=0.5), duration=dur))

# Track 3: Voiceover and captions (start after intro)
vo_track = Track()
vo_track.add_clip(start=intro_duration, clip=Clip(asset=AudioAsset(id=voiceover.id, volume=1.0), duration=voiceover_length_float))
vo_track.add_clip(start=intro_duration, clip=Clip(asset=caption_asset, duration=voiceover_length_float))

# Track 4: Subject videos - using paired transcript timestamps with looping (2x)
subject_track = Track()

# Sort by start time
sorted_subjects = sorted(subject_videos, key=lambda x: float(x['start']))

for i, sv in enumerate(sorted_subjects):
    # Add intro offset to transcript timestamps
    clip_start = intro_duration + sv['start']
    clip_duration = sv['end'] - sv['start']

    # Ensure minimum duration
    clip_duration = max(1, clip_duration)

    # Don't let this clip overlap with the next one
    if i < len(sorted_subjects) - 1:
        next_start = intro_duration + sorted_subjects[i + 1]['start']
        max_duration = next_start - clip_start
        clip_duration = min(clip_duration, max_duration)

    # Don't exceed voiceover end
    vo_end = intro_duration + voiceover_length_float
    if clip_start + clip_duration > vo_end:
        clip_duration = vo_end - clip_start

    if clip_duration <= 0:
        print(f"⚠️ Skipping subject {i+1} - no time available")
        continue

    # Use the actual length of the generated subject video for looping
    subject_video_actual_length = sv['video'].length
    num_loops = math.ceil(clip_duration / subject_video_actual_length)

    for loop in range(num_loops):
        loop_start = clip_start + (loop * subject_video_actual_length)
        remaining = clip_duration - (loop * subject_video_actual_length)
        loop_duration = min(subject_video_actual_length, remaining)

        if loop_duration <= 0:
            break

        subject_track.add_clip(
            start=loop_start,
            clip=Clip(
                asset=VideoAsset(id=sv['video'].id, volume=0),
                duration=loop_duration,
                fit=Fit.contain,
                scale=0.75,
                position=Position.top,
                offset=Offset(x=0, y=0.05),
                opacity=0.95
            )
        )

    print(f"✓ {sv['subject']}: {clip_start:.1f}s - {clip_start + clip_duration:.1f}s ({num_loops} loops)")

print(f"✅ Added {len(sorted_subjects)} subject videos with looping")

# Track 5: Title card (intro) - BIGGER, QUICKSAND FONT
title_track = Track()
title_track.add_clip(
    start=0,
    clip=Clip(
        asset=TextAsset(
            text=video_title,
            font=Font(
                family="Quicksand",
                size=96,
                color="#FFD700"
            ),
            border=Border(
                color="#000000",
                width=6
            )
        ),
        duration=intro_duration,
        position=Position.center,
        transition=Transition(in_="fade", out="fade", duration=0.8)
    )
)

# Track 6: Outro card - BIGGER, QUICKSAND FONT
outro_track = Track()
outro_track.add_clip(
    start=intro_duration + voiceover_length_float,
    clip=Clip(
        asset=TextAsset(
            text="Happy Learning!",
            font=Font(
                family="Quicksand",
                size=88,
                color="#90EE90"
            ),
            border=Border(
                color="#000000",
                width=6
            )
        ),
        duration=outro_duration,
        position=Position.center,
        transition=Transition(in_="fade", out="fade", duration=0.8)
    )
)

# Add all tracks
final_timeline.add_track(bg_video_track)
final_timeline.add_track(bg_music_track)
final_timeline.add_track(vo_track)
final_timeline.add_track(subject_track)
final_timeline.add_track(title_track)
final_timeline.add_track(outro_track)

print("✅ Final timeline built with all components!")

# Generate stream
print("\n🚀 Generating final stream...")
final_stream = final_timeline.generate_stream()
play_stream(final_stream)

🎬 Building FINAL video with subject videos...
⏱️ Total: 95.263673s (intro:5 + vo:85.263673 + outro:5)
✓ Smiling sun with orbiting planet family: 5.0s - 10.0s (2 loops)
✓ Warm lamp-like sun with toy planets circling: 10.0s - 20.0s (3 loops)
✓ Three simple planets: tiny hot, blue windy, and Earth: 20.0s - 30.0s (3 loops)
✓ Earth with nearby red Mars and a giant gas balloon planet: 30.0s - 40.0s (3 loops)
✓ Ringed planet leading a dance of colorful planets and tiny rocks: 40.0s - 50.0s (3 loops)
✓ Sun as a gentle anchor with a comet streaking by: 50.0s - 60.0s (3 loops)
✓ Cute rocket with astronaut among quiet stars: 60.0s - 70.0s (3 loops)
✓ Child gazing at a playful solar system playground: 70.0s - 80.0s (3 loops)
✓ Child astronaut waving at the moon and stars: 80.0s - 90.3s (3 loops)
✅ Added 9 subject videos with looping
✅ Final timeline built with all components!

🚀 Generating final stream...


---

## 🎊 That's a Wrap!

We just turned a simple topic into a complete animated educational video — all with code!

### What we built:
- **AI-Generated Script** — Simple enough for a 5-year-old
- **Natural Voiceover** — Friendly narration
- **Dreamy Background** — Looped animated visuals
- **Fun Music** — Whimsical background tunes
- **Subject Videos** — Cartoonish clips that pop up at the right moments
- **Custom Captions** — Kids can follow along word by word!

### The Magic of VideoDB Editor SDK:
- 🎬 No external tools — everything in Python
- 🔄 Loopable backgrounds with simple math
- 🎯 Precise timing with timestamp analysis  
- 🎨 Picture-in-picture with positioning control
- 📝 Auto-generated captions

**Want to try a different topic?** Just change the `topic` variable at the top and run it again! 🚀

---

*Made with ❤️ using VideoDB Editor SDK*